# 02. 대중교통(지하철) x 택시 수단분담률 분석

## 분석 배경 및 목적

도시 교통 체계에서 택시와 대중교통(지하철)은 보완재이자 대체재의 이중적 관계를 갖는다. 출퇴근 시간대에는 지하철이 주된 수단이므로 택시 분담률이 낮지만, 심야 시간대에는 지하철 운행이 중단되면서 택시가 사실상 유일한 대안이 된다. 이 관계를 정량적으로 파악하면 다음과 같은 질문에 답할 수 있다.

1. **시간대별 수단분담률**: 택시가 전체 이동에서 차지하는 비중이 시간대에 따라 어떻게 변하는가?
2. **막차 후 전환 효과**: 지하철 막차(약 24시) 이후 택시 수요가 얼마나 급증하는가?
3. **월별/요일별 변동**: 수단분담률의 계절성과 요일 패턴은 어떠한가?

**방법론적 근거**: ScienceDirect (2023)의 싱가포르 연구는 도시 규모 데이터를 활용하여 택시와 대중교통의 통합 수단/경로 선택 모형을 추정하였다. 해당 연구는 시간대별로 두 수단이 대체재에서 보완재로 전환되는 패턴을 실증하였으며, 본 분석은 서울의 택시-지하철 관계에서 동일한 구조적 패턴을 확인한다.

- **내부**: DC_TBYXD012 (요금정보)
- **외부**: `subway_ridership_2024.csv` (서울교통공사 역별 시간대별 승하차, 2024년)
- **주의**: 지하철 데이터는 2024년만 존재하므로 수단분담률 비교는 2024년 공통 기간으로 한정한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

## 1. 지하철 데이터 로드 (명세서 컬럼 기준)

In [ ]:
# === 경로 설정 (폐쇄망 환경에 맞게 수정) ===
D012_PATH = './DC_TBYXD012.csv'
EXT_DIR   = './external_data'
SUBWAY    = f'{EXT_DIR}/subway_ridership_2024.csv'

# 명세서 컬럼: 수송일자, 호선, 역명, 승하차구분, 06시이전, 06-07시간대 ... 24시이후
subway = pd.read_csv(SUBWAY, encoding='utf-8')
subway['수송일자'] = pd.to_datetime(subway['수송일자'])
print(f"지하철 원본: {subway.shape}")
print(f"기간: {subway['수송일자'].min().date()} ~ {subway['수송일자'].max().date()}")
print('컬럼:', subway.columns.tolist())
subway.head(2)

In [ ]:
# 시간대 컬럼 → long format. 명세서 시간대 라벨과 대표 시(hour) 매핑
time_cols = ['06시이전','06-07시간대','07-08시간대','08-09시간대','09-10시간대',
             '10-11시간대','11-12시간대','12-13시간대','13-14시간대','14-15시간대',
             '15-16시간대','16-17시간대','17-18시간대','18-19시간대','19-20시간대',
             '20-21시간대','21-22시간대','22-23시간대','23-24시간대','24시이후']
hour_map  = [5, 6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23, 0]  # 시작시 기준, 06시이전=5, 24시이후=0

# 존재하는 컬럼만 사용 (원본에 일부 누락 대비)
present = [(c, h) for c, h in zip(time_cols, hour_map) if c in subway.columns]
melt = subway.melt(id_vars=['수송일자','호선','역명'],
                   value_vars=[c for c, _ in present],
                   var_name='time_label', value_name='riders')
label2hour = {c: h for c, h in present}
melt['hour'] = melt['time_label'].map(label2hour)
melt['riders'] = pd.to_numeric(melt['riders'], errors='coerce').fillna(0)

# 일별×시간대 총 승하차 (승차+하차 모두 포함)
subway_hourly = melt.groupby(['수송일자','hour'])['riders'].sum().reset_index()
subway_hourly.columns = ['date','hour','subway_rides']
print(f"지하철 시간별: {len(subway_hourly):,}행")
subway_hourly.head()

## 2. 택시 일별·시간대별 집계 (청크, 2024년만)

In [ ]:
# 지하철이 2024년만 있으므로 택시도 2024년으로 한정해 공정 비교
usecols = ['RIDE_DTIME']
dtypes  = {'RIDE_DTIME': str}

daily, hourly, dow = {}, {}, {}
total = filt = 0
for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    total += len(chunk)
    m = chunk['RIDE_DTIME'].str[:4] == '2024'
    c24 = chunk[m]
    if len(c24):
        d = c24['RIDE_DTIME'].str[:8]
        h = c24['RIDE_DTIME'].str[8:10].astype(int)
        wd = pd.to_datetime(d, format='%Y%m%d').dt.dayofweek
        for k, v in c24.groupby([d, h]).size().items():
            hourly[k] = hourly.get(k, 0) + v
        for k, v in c24.groupby(d).size().items():
            daily[k] = daily.get(k, 0) + v
        for k, v in c24.groupby([wd, h]).size().items():
            dow[k] = dow.get(k, 0) + v
        filt += len(c24)
    del chunk
    gc.collect()

taxi_hourly = (pd.Series(hourly).rename_axis(['date','hour']).reset_index(name='taxi_rides'))
taxi_hourly['date'] = pd.to_datetime(taxi_hourly['date'], format='%Y%m%d')
taxi_daily = (pd.Series(daily).rename_axis('date').reset_index(name='taxi_rides'))
taxi_daily['date'] = pd.to_datetime(taxi_daily['date'], format='%Y%m%d')
taxi_dow = (pd.Series(dow).rename_axis(['weekday','hour']).reset_index(name='taxi_rides'))
del daily, hourly, dow; gc.collect()

print(f"전체 {total:,}행 중 2024년 {filt:,}행")
print(f"택시 일별 {len(taxi_daily)}일, 시간별 {len(taxi_hourly)}행")
mem_usage('after load')

## 3. 시간대별 택시 vs 지하철 + 택시 분담률

시간대별 택시/(택시+지하철) 비율을 산출한다. 이 단순 분담률은 두 수단 간 경쟁/보완 관계의 시간적 구조를 드러낸다. ScienceDirect (2023)의 싱가포르 연구에서도 동일한 시간대별 분담률 분석을 수행하였으며, 심야와 새벽에 택시 분담률이 급격히 상승하는 보편적 패턴을 보고하였다.

In [ ]:
# 시간대별 평균 (2024 공통 기간)
t_h = taxi_hourly.groupby('hour')['taxi_rides'].mean()
s_h = subway_hourly.groupby('hour')['subway_rides'].mean()
compare = pd.concat([t_h.rename('taxi'), s_h.rename('subway')], axis=1).fillna(0)
compare['taxi_share_pct'] = (compare['taxi'] / (compare['taxi'] + compare['subway']) * 100).round(2)
compare = compare.reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
axt = ax1.twinx()
ax1.bar(compare['hour'] - 0.2, compare['subway'], width=0.4, color='#4CAF50', alpha=0.7, label='지하철')
axt.bar(compare['hour'] + 0.2, compare['taxi'], width=0.4, color='#FF9800', alpha=0.7, label='택시')
ax1.set_ylabel('지하철 일평균 승하차', color='#4CAF50')
axt.set_ylabel('택시 일평균 승차', color='#FF9800')
ax1.set_title('시간대별 지하철 vs 택시', fontweight='bold'); ax1.set_xticks(range(24))
ax1.legend(loc='upper left'); axt.legend(loc='upper right'); ax1.grid(alpha=0.3)

colors = ['#F44336' if v > 50 else '#FF9800' if v > 20 else '#2196F3' for v in compare['taxi_share_pct']]
ax2.bar(compare['hour'], compare['taxi_share_pct'], color=colors)
ax2.axhline(50, color='red', ls='--', alpha=0.5)
ax2.set_xlabel('시간대'); ax2.set_ylabel('택시 분담률 (%)')
ax2.set_title('시간대별 택시 수단분담률', fontweight='bold'); ax2.set_xticks(range(24)); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
compare

## 4. 막차 후 택시 수요 전환

지하철 막차 시간(약 23:30~00:00) 전후의 택시 수요 변화를 분석한다. 막차 후 대중교통 대안이 사라지면서 택시로의 수요 전환(mode shift)이 발생하는데, 이 전환의 크기와 지속 시간을 정량화한다. 이는 심야 택시 공급 계획과 심야버스 노선 설계의 근거 자료가 된다.

In [ ]:
night = [22, 23, 0, 1, 2, 3, 4]
order = {22:0, 23:1, 0:2, 1:3, 2:4, 3:5, 4:6}
nt = compare[compare['hour'].isin(night)].copy()
nt['o'] = nt['hour'].map(order); nt = nt.sort_values('o')
labels = ['22시','23시','0시','1시','2시','3시','4시']

fig, ax = plt.subplots(figsize=(12, 6))
axt = ax.twinx()
x = range(len(labels))
ax.bar([i-0.2 for i in x], nt['subway'].values, width=0.4, color='#4CAF50', alpha=0.7, label='지하철')
axt.bar([i+0.2 for i in x], nt['taxi'].values, width=0.4, color='#FF9800', alpha=0.7, label='택시')
ax.axvline(1.5, color='red', ls='--', lw=2, label='지하철 막차(약 23:30~00:00)')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('지하철', color='#4CAF50'); axt.set_ylabel('택시', color='#FF9800')
ax.set_title('막차 후 택시 수요 전환', fontweight='bold')
ax.legend(loc='upper left'); axt.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

pre = compare.loc[compare['hour']==22, 'taxi'].values[0]
post = compare.loc[compare['hour']==0, 'taxi'].values[0]
print(f"22시 택시: {pre:,.0f} → 0시 택시: {post:,.0f} ({(post/pre-1)*100:+.1f}%)")

## 5. 월별 수단분담률 추이 (2024)

월별 분담률 변동은 계절성(여름 휴가, 겨울 한파)과 이벤트(명절, 연말) 효과를 반영한다. 분담률의 월별 추이를 통해 택시 수요가 지하철 대비 상대적으로 증가하는 시기를 식별할 수 있다.

In [ ]:
tm = taxi_daily.assign(ym=taxi_daily['date'].dt.to_period('M').astype(str)).groupby('ym')['taxi_rides'].sum()
sm = subway_hourly.assign(ym=subway_hourly['date'].dt.to_period('M').astype(str)).groupby('ym')['subway_rides'].sum()
mc = pd.concat([tm.rename('taxi'), sm.rename('subway')], axis=1).dropna().reset_index()
mc['taxi_share'] = mc['taxi'] / (mc['taxi'] + mc['subway']) * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(mc['ym'], mc['taxi_share'], 'o-', color='#2196F3', lw=2)
ax.fill_between(range(len(mc)), mc['taxi_share'], alpha=0.2, color='#2196F3')
ax.set_ylabel('택시 분담률 (%)'); ax.set_title('월별 택시 수단분담률 추이 (2024)', fontweight='bold')
plt.xticks(rotation=45); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
mc

## 6. 호선별 심야 비중

In [ ]:
# 호선별 총 승하차 및 심야(22~04시) 비중
line_h = melt.copy()
line_h['is_night'] = line_h['hour'].isin([22, 23, 0, 1, 2, 3, 4])
line_stats = line_h.groupby('호선').agg(
    total=('riders', 'sum'),
    night=('riders', lambda s: s[line_h.loc[s.index, 'is_night']].sum())).reset_index()
line_stats['night_pct'] = (line_stats['night'] / line_stats['total'] * 100).round(1)
line_stats = line_stats.sort_values('total', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.barh(line_stats['호선'], line_stats['total'], color='#4CAF50', alpha=0.7)
ax1.set_title('호선별 총 승하차', fontweight='bold'); ax1.invert_yaxis(); ax1.grid(alpha=0.3, axis='x')
ax2.barh(line_stats['호선'], line_stats['night_pct'], color='#FF9800', alpha=0.7)
ax2.set_title('호선별 심야(22~04시) 비중 (%)', fontweight='bold'); ax2.invert_yaxis(); ax2.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
line_stats

## 7. 요일별 택시 수요

In [ ]:
dow_total = taxi_dow.groupby('weekday')['taxi_rides'].sum()
dow_labels = ['월','화','수','목','금','토','일']
colors = ['#2196F3']*4 + ['#FF9800'] + ['#F44336']*2
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(7), [dow_total.get(i, 0) for i in range(7)], color=colors)
ax.set_xticks(range(7)); ax.set_xticklabels(dow_labels)
ax.set_ylabel('2024 누적 택시 승차'); ax.set_title('요일별 택시 수요 (2024)', fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 8. 요약

본 분석의 핵심 발견과 실무적 시사점을 정리한다.

**실무 활용**: 시간대별 수단분담률은 심야버스 노선 확충, 심야할증 정책, 택시 공급 조절의 정량적 근거로 활용할 수 있다. 특히 막차 후 전환 수요의 크기는 심야 자율주행 택시(robo-taxi) 도입 시 필요 차량 수 추정의 기초 데이터가 된다.

In [ ]:
peak = compare.loc[compare['taxi_share_pct'].idxmax()]
low  = compare.loc[compare['taxi_share_pct'].idxmin()]
day  = compare[compare['hour'].between(6, 22)]['taxi_share_pct'].mean()
ngt  = compare[compare['hour'].isin([23,0,1,2,3,4,5])]['taxi_share_pct'].mean()
print('=' * 60)
print('대중교통 × 택시 수단분담률 요약 (2024 기준)')
print('=' * 60)
print(f"택시 분담률 최고: {int(peak['hour'])}시 ({peak['taxi_share_pct']:.1f}%)")
print(f"택시 분담률 최저: {int(low['hour'])}시 ({low['taxi_share_pct']:.1f}%)")
print(f"주간(06~22) 평균: {day:.1f}% / 심야(23~05) 평균: {ngt:.1f}% → 심야가 {ngt/day:.1f}배")

---

## References

1. Qin, H., Gao, J., Koh, J. E. H., & Ong, G. P. (2023). Integrated taxi and transit mode and route choice using city-scale data. *Transportation Research Part A: Policy and Practice*, 175, 103782. (ScienceDirect)
2. Ben-Akiva, M. E., & Lerman, S. R. (1985). *Discrete Choice Analysis: Theory and Application to Travel Demand*. MIT Press.
3. 서울교통공사 (2024). 역별 시간대별 승하차인원 통계. https://data.seoul.go.kr